# Create or append Virtual Icechunk Store from SHYFEM forecast NetCDF files

* This notebook appends Taranto SHYFEM forecast data to an Icechunk store in the **rswain** bucket using date-based set logic.
* Source NetCDF files are read from `protocoast-data`; the Icechunk repo is written to `rswain`.
* If no repo exists, a new one will be created with references to all the existing NetCDF files. 

**Append Methodology:**
1. **`set_repo`**: extract all dates currently present in the Icechunk store's `time` coordinate
2. **`set_cloud`**: Scan the S3 bucket for all available NOS files and extract their dates.
3. **`new_dates`**: Calculate the difference (`set_cloud - set_repo`) to determine exactly which days need to be written for creation or appended.

In [1]:
import warnings
import os
import pandas as pd
import fsspec
import xarray as xr
from dotenv import load_dotenv

warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
import icechunk
from obstore.store import from_url
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import HDFParser
from obspec_utils.registry import ObjectStoreRegistry
from obstore.store import S3Store

In [3]:
# Load credentials
_ = load_dotenv(f'{os.environ["HOME"]}/dotenv/protocoast.env', override=True)

# Configuration
storage_endpoint = os.environ['ENDPOINT_URL']
icechunk_bucket = 'rswain'           # bucket where the Icechunk repo is stored
source_bucket = 'protocoast-data'    # bucket where source NetCDF files live
storage_name = 'taranto-icechunk-tubitak-v1'
icechunk_bucket_url = f"s3://{icechunk_bucket}"
source_bucket_url = f"s3://{source_bucket}"

# Setup Filesystem (for scanning source files)
fs = fsspec.filesystem('s3', anon=False, endpoint_url=storage_endpoint,
                       skip_instance_cache=True, use_listings_cache=False)

In [4]:
# Define Icechunk Storage & Config
# Repo is written to the rswain bucket
storage = icechunk.s3_storage(
    bucket=icechunk_bucket,
    prefix=f"icechunk/{storage_name}",
    from_env=True,
    endpoint_url=storage_endpoint,
    region='not-used',
    force_path_style=True,
)

config = icechunk.RepositoryConfig.default()
# Virtual chunks point back to the source data in protocoast-data
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=f"{source_bucket_url}/",
        store=icechunk.s3_store(region="not-used", anonymous=False, s3_compatible=True,
                                force_path_style=True, endpoint_url=storage_endpoint),
    ),
)

credentials = icechunk.containers_credentials({f"{source_bucket_url}/": icechunk.s3_credentials(anonymous=False)})

store_obj = S3Store(
    bucket=source_bucket,
    endpoint=storage_endpoint,
    region="not-used",
)

registry = ObjectStoreRegistry({source_bucket_url: store_obj})
parser = HDFParser()

### Step 1: Create Date Sets (`set_repo` vs `set_cloud`)

In [5]:
# --- 1. Get Dates from Icechunk Repo (set_repo) ---
try:
    repo = icechunk.Repository.open(storage, config, authorize_virtual_chunk_access=credentials)
    session = repo.readonly_session("main")
    ds = xr.open_zarr(session.store, consolidated=False, chunks={})
    
    if 'time' in ds.coords:
        # Extract dates as YYYYMMDD strings
        dates = pd.to_datetime(ds.time.values) + pd.Timedelta(days=1)
        set_repo = set(dates.strftime('%Y%m%d'))
    else:
        set_repo = set()
        
except Exception as e:
    print(f"Repo access failed or empty ({e}). Assuming set_repo is empty.")
    repo = None
    set_repo = set()

print(f"set_repo: {len(set_repo)} dates found.")

# --- 2. Get Dates from Cloud Bucket (set_cloud) ---
print("Scanning S3 for NOS files...")
nos_files = fs.glob(f'{source_bucket}/full_dataset/shyfem/taranto/forecast/*/*nos*.nc')

set_cloud = set()
date_to_files_map = {}

for f in nos_files:
    try:
        date_str = f.split('/')[-2]
        set_cloud.add(date_str)
        
        base_dir = os.path.dirname(f)
        nos_path = f's3://{f}'
        ous_path = f's3://{base_dir}/taranto_ous_{date_str}_nc4.nc'
        
        date_to_files_map[date_str] = {'nos': nos_path, 'ous': ous_path}
        
    except IndexError:
        pass

print(f"set_cloud: {len(set_cloud)} dates found.")

Repo access failed or empty (  x the repository doesn't exist
  | 
  | context:
  |    0: icechunk::repository::open
  |              at icechunk/src/repository.rs:343
  | 
  `-> the repository doesn't exist
). Assuming set_repo is empty.
set_repo: 0 dates found.
Scanning S3 for NOS files...
set_cloud: 202 dates found.


In [6]:
# --- 3. Determine New Dates ---
new_dates = sorted(list(set_cloud - set_repo))

print(f"Dates to process: {len(new_dates)}")
if new_dates:
    print(f"Range: {new_dates[0]} to {new_dates[-1]}")

Dates to process: 202
Range: 20250909 to 20260420


In [13]:
new_dates = new_dates[:3]
new_dates

['20260416', '20260417', '20260418']

### Step 2: Virtualize and Merge New Data

In [15]:
def fix_ds(ds):
    """Standardizes dimensions and coordinates for Taranto dataset."""
    ds = ds.rename_vars(time='valid_time')
    ds = ds.rename_dims(time='step')
    step = (ds.valid_time - ds.valid_time[0]).assign_attrs({"standard_name": "forecast_period"})
    time = ds.valid_time[0].assign_attrs({"standard_name": "forecast_reference_time"})
    ds = ds.assign_coords(step=step, time=time)
    ds = ds.drop_indexes("valid_time")
    ds = ds.drop_vars('valid_time')
    ds = ds.set_coords(['latitude', 'longitude', 'element_index', 'topology', 'total_depth'])
    return ds

In [16]:
ds_final = None

# new_dates = new_dates[:3]  # only if testing

if new_dates:
    nos_urls = [date_to_files_map[d]['nos'] for d in new_dates]
    ous_urls = [date_to_files_map[d]['ous'] for d in new_dates]

    # --- Process NOS ---
    print(f"Virtualizing {len(nos_urls)} NOS files...")
    nos_list = [
        open_virtual_dataset(url, parser=parser, registry=registry, loadable_variables=["time"])
        for url in nos_urls
    ]
    nos_list = [fix_ds(ds) for ds in nos_list]
    combined_nos = xr.concat(
        nos_list, dim="time", coords="minimal", compat="override", combine_attrs="override"
    )

    # --- Process OUS ---
    print(f"Virtualizing {len(ous_urls)} OUS files...")
    ous_list = [
        open_virtual_dataset(url, parser=parser, registry=registry, loadable_variables=["time"])
        for url in ous_urls
    ]
    ous_list = [fix_ds(ds) for ds in ous_list]
    combined_ous = xr.concat(
        ous_list, dim="time", coords="minimal", compat="override", combine_attrs="override"
    )

    # --- Merge ---
    ds_final = xr.merge([combined_nos, combined_ous], compat='override')
    print("Datasets merged and ready for writing to Icechunk.")
    
else:
    print("No new dates found. Skipping processing.")

Virtualizing 3 NOS files...
Virtualizing 3 OUS files...
Datasets merged and ready for writing to Icechunk.


### Step 3: Append to Icechunk

In [17]:
if ds_final is not None:
    if repo is None:
        repo = icechunk.Repository.create(storage, config, authorize_virtual_chunk_access=credentials)
        initial_session = repo.writable_session("main")

        print(f"Writing {len(ds_final.time)} time steps to Icechunk...")
        ds_final.virtualize.to_icechunk(initial_session.store)
    
        msg = f"Initialized with forecast data: {new_dates[0]} to {new_dates[-1]}"
        initial_session.commit(msg)
        print(f"Commit successful: '{msg}'")
    else:
        append_session = repo.writable_session("main")

        print(f"Appending {len(ds_final.time)} time steps to Icechunk...")
        ds_final.virtualize.to_icechunk(append_session.store, append_dim="time")
    
        msg = f"Appended forecast data: {new_dates[0]} to {new_dates[-1]}"
        append_session.commit(msg)
        print(f"Commit successful: '{msg}'")

    history = repo.ancestry(branch="main")
    latest = next(history)
    print(f"Latest Commit [{latest.written_at}]: {latest.message}")
    
else:
    print("Nothing to append.")

Writing 3 time steps to Icechunk...
Commit successful: 'Initialized with forecast data: 20260416 to 20260418'
Latest Commit [2026-04-21 14:37:46.883212+00:00]: Initialized with forecast data: 20260416 to 20260418


In [ ]:
nos_files

In [22]:
xr.open_dataset(fs.open(nos_files[0]))

<xarray.Dataset> Size: 2GB
Dimensions:        (element: 58285, vertex: 3, node: 30731, time: 144, level: 70)
Coordinates:
  * level          (level) float32 280B 2.0 4.0 6.0 ... 1.5e+03 2e+03 2.5e+03
  * time           (time) datetime64[ns] 1kB 2025-09-08 ... 2025-09-13T23:00:00
Dimensions without coordinates: element, vertex, node
Data variables:
    element_index  (element, vertex) int32 699kB ...
    latitude       (node) float32 123kB ...
    longitude      (node) float32 123kB ...
    salinity       (time, node, level) float32 1GB ...
    temperature    (time, node, level) float32 1GB ...
    topology       int32 4B ...
    total_depth    (node) float32 123kB ...
Attributes:
    Conventions:  CF-1.4
    title:        uae
    history:      Wed Sep 10 15:12:42 2025: ncks -4 -L 5 -O -d time,0,143 --c...
    institution:  Centro Euro-Mediterraneo sui Cambiamenti Climatici - CMCC, ...
    source:       Model data produced by SHYFEM-MPI at CMCC
    references:   https://zenodo.org/record/5596734#.Y-YwpxPMLx8
    NCO:          netCDF Operators version 5.0.6 (Homepage = http://nco.sf.ne...

In [23]:
ds_final

<xarray.Dataset> Size: 15GB
Dimensions:        (time: 3, step: 144, node: 30731, level: 70, element: 58285,
                    vertex: 3)
Coordinates:
    element_index  (element, vertex) int32 699kB ManifestArray<shape=(58285, ...
    latitude       (node) float32 123kB ManifestArray<shape=(30731,), dtype=f...
    longitude      (node) float32 123kB ManifestArray<shape=(30731,), dtype=f...
    topology       int32 4B ManifestArray<shape=(), dtype=int32, chunks=()>
    total_depth    (node) float32 123kB ManifestArray<shape=(30731,), dtype=f...
    level          (level) float32 280B ManifestArray<shape=(70,), dtype=floa...
  * step           (step) timedelta64[ns] 1kB 00:00:00 ... 5 days 23:00:00
  * time           (time) datetime64[ns] 24B 2026-04-15 2026-04-16 2026-04-17
Dimensions without coordinates: node, element, vertex
Data variables:
    salinity       (time, step, node, level) float32 4GB ManifestArray<shape=...
    temperature    (time, step, node, level) float32 4GB ManifestArray<shape=...
    u_velocity     (time, step, node, level) float32 4GB ManifestArray<shape=...
    v_velocity     (time, step, node, level) float32 4GB ManifestArray<shape=...
    water_level    (time, step, node) float32 53MB ManifestArray<shape=(3, 14...
Attributes:
    Conventions:  CF-1.4
    title:        uae
    history:      Thu Apr 16 16:50:12 2026: ncks -4 -L 5 -O -d time,0,143 --c...
    institution:  Centro Euro-Mediterraneo sui Cambiamenti Climatici - CMCC, ...
    source:       Model data produced by SHYFEM-MPI at CMCC
    references:   https://zenodo.org/record/5596734#.Y-YwpxPMLx8
    NCO:          netCDF Operators version 5.0.6 (Homepage = http://nco.sf.ne...